# 03 - Feature Extraction

In [ ]:
import os, sys, zipfile
from pathlib import Path

# Cari root repo: di laptop = folder di atas notebooks/, di Kaggle = dataset mlc-src
if os.path.exists("/kaggle/input"):
    hits = list(Path("/kaggle/input").rglob("src/data.py"))
    if hits:
        ROOT = hits[0].parent.parent
    else:  # dataset masih berupa zip
        ROOT = Path("/kaggle/working/_repo")
        for z in Path("/kaggle/input").rglob("*.zip"):
            if z.stem in ("src", "configs", "annotations"):
                zipfile.ZipFile(z).extractall(ROOT / z.stem)
else:
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.data import ON_KAGGLE, get_path, load_config
from src.seed import seed_everything

cfg = load_config()
seed_everything(cfg["SEED"])
print("Kaggle:", ON_KAGGLE, "| data:", get_path(cfg, "raw"))

Mengubah tiap frame menjadi 768 angka memakai encoder beku (tanpa training).

- Train: frame diperkecil ke 640x360 lalu bolak-balik JPEG (kualitas acak 70-95), sesuai FAQ 73.
- Test: semua 30 frame diproses langsung, satu sequence demi satu sequence (FAQ 66, 70).
- Tidak ada statistik yang dihitung dari data test (FAQ 38, 68).

In [ ]:
import json, time
import numpy as np
import torch

from src import features as F

# Encoder beku (tanpa training): DINOv2 kecil, bobot pretrained LVD (bukan data eksternal: FAQ 3, 15, 22).
ENCODER = "vit_small_patch14_dinov2.lvd142m"
SEED = cfg["SEED"]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# Di Kaggle tanpa GPU, ekstraksi bisa makan berjam-jam. Lebih baik berhenti dan perbaiki setelan.
assert DEVICE == "cuda" or not ON_KAGGLE, (
    "GPU belum aktif. Jalankan lewat kaggle_run.py tanpa --cpu, dan pastikan akun sudah verifikasi telepon."
)
print("device:", DEVICE, "| encoder:", ENCODER)

raw = get_path(cfg, "raw")
out_dir = get_path(cfg, "features")          # ditulis ke /kaggle/working/features di Kaggle
emb_train_dir = out_dir / "emb_train"
print("fitur disimpan ke:", out_dir)

In [ ]:
enc = F.buat_encoder(ENCODER, device=DEVICE, pretrained=True)
D = F.dim_fitur(enc)
print("dimensi fitur per frame:", D)

In [ ]:
# Semua video train. Nama file keluaran: <kategori>__<subject>__<nama video>.npy
def nama_fitur(p):
    p = Path(p)
    return f"{p.parent.parent.name}__{p.parent.name}__{p.stem}"

videos = sorted(
    p for kategori in ("normal", "cheat")
    for p in (raw / "train" / kategori).glob("subject_*/*") if p.suffix.lower() == ".mov"
)
print("jumlah video train:", len(videos))
assert len(videos) > 0, "Tidak ada video train ditemukan"

t0 = time.time()
F.ekstrak_banyak_video(enc, videos, emb_train_dir, seed=SEED, nama_fn=nama_fitur)
print(f"selesai video train dalam {(time.time() - t0) / 60:.1f} menit")

In [ ]:
# Semua sequence test: 30 frame diproses LANGSUNG dan tiap sequence sendiri-sendiri (FAQ 66, 70, 38).
emb_test_path = out_dir / "emb_test.npy"
ids_test_path = out_dir / "emb_test_ids.txt"
if emb_test_path.exists() and ids_test_path.exists():
    print("fitur test sudah ada, dilewati")
else:
    t0 = time.time()
    ids, feats = F.ekstrak_test(enc, raw / "test")
    np.save(emb_test_path, feats)
    ids_test_path.write_text("\n".join(ids), encoding="utf-8")
    print(f"selesai test dalam {(time.time() - t0) / 60:.1f} menit | {feats.shape}")

In [ ]:
# Pemeriksaan hasil
npy = sorted(emb_train_dir.glob("*.npy"))
panjang = [np.load(p, mmap_mode="r").shape[0] for p in npy]
print("video train berfitur:", len(npy), "| frame min/maks:", min(panjang), max(panjang))
assert len(npy) == len(videos), "Ada video yang belum terekstrak"

et = np.load(emb_test_path, mmap_mode="r")
ids = ids_test_path.read_text(encoding="utf-8").split("\n")
print("test:", et.shape, "| jumlah id:", len(ids))
assert et.shape[0] == len(ids) and et.shape[1] == 30 and et.shape[2] == D
assert np.isfinite(np.asarray(et[:50], dtype=np.float32)).all(), "Ada NaN/inf di fitur test"
assert np.isfinite(np.load(npy[0]).astype(np.float32)).all(), "Ada NaN/inf di fitur train"

meta = {"encoder": ENCODER, "dim": int(D), "seed": SEED, "kualitas_jpeg": list(F.KUALITAS_JPEG),
        "ukuran_input": [F.TINGGI_INPUT, F.LEBAR_INPUT], "n_video_train": len(npy), "n_seq_test": len(ids)}
(out_dir / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print(meta)